# Task 1: Document Chunking & Embeddings

Corpus: Flask 3.1.0 docs (`.rst` sources from the `pallets/flask` repo).

Steps:
1. Get the raw corpus
2. **Clean the reStructuredText** (this notebook section)
3. Chunk (200 words, 20 overlap)
4. Embed with all-MiniLM-L6-v2 and verify shape (N, 384)

In [1]:
%load_ext autoreload
%autoreload 2

import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

from raghood.scrape import ROOT, RAW_DIR, scrape_all

PROCESSED_DIR = ROOT / "data" / "processed"

## 1. Get the raw corpus


In [ ]:
manifest = scrape_all()   # ~2 s; overwrites data/raw
df = pd.DataFrame(manifest)
print(f"{len(df)} pages, {df.words.sum():,} words")
df.sort_values("words", ascending=False).head(8)[["page", "words", "url"]]

71 pages, 59,570 words


,page,words,url
50,quickstart,4377,https://flask.palletsprojects.com/en/3.1.x/qui...
5,config,3637,https://flask.palletsprojects.com/en/3.1.x/con...
0,api,2593,https://flask.palletsprojects.com/en/3.1.x/api/
19,errorhandling,2357,https://flask.palletsprojects.com/en/3.1.x/err...
4,cli,2212,https://flask.palletsprojects.com/en/3.1.x/cli/
67,tutorial/tests,2149,https://flask.palletsprojects.com/en/3.1.x/tut...
20,extensiondev,1783,https://flask.palletsprojects.com/en/3.1.x/ext...
18,design,1774,https://flask.palletsprojects.com/en/3.1.x/des...


## 2. What does the raw text look like?

It is reStructuredText, not plain prose. Embedding models never saw `:class:` or `.. code-block::`,
so leaving the markup in adds noise to every chunk. Here is a slice of a real page:

In [3]:
raw = (RAW_DIR / "blueprints.rst").read_text(encoding="utf-8")
print(raw[:1200])

Modular Applications with Blueprints

.. currentmodule:: flask

.. versionadded:: 0.7

Flask uses a concept of *blueprints* for making application components and
supporting common patterns within an application or across applications.
Blueprints can greatly simplify how large applications work and provide a
central means for Flask extensions to register operations on applications.
A :class:`Blueprint` object works similarly to a :class:`Flask`
application object, but it is not actually an application.  Rather it is a
*blueprint* of how to construct or extend an application.

Why Blueprints?
---------------

Blueprints in Flask are intended for these cases:

* Factor an application into a set of blueprints.  This is ideal for
  larger applications; a project could instantiate an application object,
  initialize several extensions, and register a collection of blueprints.
* Register a blueprint on an application at a URL prefix and/or subdomain.
  Parameters in the URL prefix/subdomain b

Markup that appears in the corpus (counts over all 71 pages):

In [4]:
all_raw = "\n".join(p.read_text(encoding="utf-8") for p in RAW_DIR.rglob("*.rst"))
directives = Counter(re.findall(r"^\s*\.\. ([\w:-]+)::", all_raw, re.M))
roles = Counter(re.findall(r":([\w:-]+):`", all_raw))
print("directives:", dict(directives.most_common(12)))
print("roles     :", dict(roles.most_common(10)))

directives: {'code-block': 255, 'versionadded': 32, 'py:data': 29, 'autofunction': 26, 'group-tab': 24, 'autoclass': 23, 'currentmodule': 20, 'data': 18, 'sourcecode': 18, 'admonition': 13, 'versionchanged': 11, 'tabs': 8}
roles     : {'meth': 135, 'doc': 130, 'class': 103, 'data': 101, 'func': 82, 'attr': 66, 'file': 50, 'ref': 23, 'exc': 22, 'mod': 8}


## 3. Cleaning rules

| Markup | What we do | Why |
|---|---|---|
| `Title` + `=====` underline | `# Title` (level by order of first appearance) | Keeps section structure for citations and eval labels |
| `.. code-block::`, `.. sourcecode::`, `text::` literal blocks | Fenced ` ``` ` block, contents untouched | Flask questions often hinge on code |
| `:class:`~flask.Flask``, `:meth:`...`, `:doc:`text <page>`` | Just the readable name (`Flask`) | Roles are Sphinx syntax; the name is the useful part |
| ` ``code`` `, `` `Click`_ `` links | `` `code` ``, `Click` | Plain text |
| `.. note::`, `.. warning::`, `.. admonition::` | `Note: ...` etc., body kept | Real content |
| `.. versionadded:: 0.7` | `Added in version 0.7.` | Real content |
| `.. py:data:: DEBUG` (+ body) | Name line + description | Config docs live here |
| `.. tabs::` / `.. tab::` | Drop marker, keep content | Layout only |
| `.. autofunction::`, `.. autoclass::` | **Drop** | Docstrings are not in the `.rst`, so these are bare names with no content that would only match keyword searches |
| `.. toctree::`, `.. currentmodule::`, `.. image::`, link targets, `----` | Drop | Navigation / layout only |

In [5]:
DROP_DIRECTIVES = {"currentmodule", "module", "toctree", "image", "rst-class",
                   "highlight", "include", "literalinclude", "only", "contents", "index"}
KEEP_BODY_DIRECTIVES = {"tabs", "group-tab", "tab"}  # marker is layout only; the content matters
AUTODOC_DIRECTIVES = {"autofunction", "autoclass", "autodata", "automodule"}  # docstrings are not in the .rst
REFERENCE_DIRECTIVES = {"py:data", "data", "attribute", "function", "class", "method", "py:function", "py:class"}
CODE_DIRECTIVES = {"code-block", "sourcecode", "code"}
ADMONITIONS = {"note": "Note", "warning": "Warning", "tip": "Tip", "important": "Important",
               "danger": "Danger", "attention": "Attention", "caution": "Caution", "seealso": "See also"}
VERSION_DIRECTIVES = {"versionadded": "Added in version", "versionchanged": "Changed in version",
                      "deprecated": "Deprecated since version"}

DIRECTIVE = re.compile(r"^(\s*)\.\. ([\w:-]+)::\s*(.*)$")
UNDERLINE = re.compile(r"^([=\-~^\"`#*+])\1{2,}\s*$")
ROLE = re.compile(r":([\w:-]+):`([^`]+)`")


def _role(m: re.Match) -> str:
    """:class:`~flask.Flask` -> Flask ; :doc:`text <page>` -> text"""
    role, target = m.group(1), m.group(2)
    if role == "gh":
        return f"issue {target}"
    titled = re.match(r"^(.*?)\s*<([^>]+)>$", target, re.S)
    if titled:
        return titled.group(1) or titled.group(2)
    if target.startswith("~"):
        return target.lstrip("~").split(".")[-1]
    return target.lstrip(".!/")


def _inline(t: str) -> str:
    t = ROLE.sub(_role, t)
    t = re.sub(r"`([^`<]+?)\s*<(https?://[^>]+)>`_{1,2}", r"\1", t)  # `text <url>`_ -> text
    t = re.sub(r"(?<!`)`([^`\n]+)`_{1,2}(?!\w)", r"\1", t)            # `Click`_ -> Click
    t = re.sub(r"``([^`]+)``", r"`\1`", t)                             # ``code`` -> `code`
    t = re.sub(r"\|\w+\|", "", t)                                      # |substitution| refs
    return t


def _indent(line: str) -> int:
    return len(line) - len(line.lstrip(" "))


def _block(lines: list[str], start: int, parent_indent: int):
    """Collect the indented body after a directive / '::' marker.
    Returns (body lines dedented, index of first line after the block)."""
    i = start
    while i < len(lines) and re.match(r"^\s+:[\w-]+:", lines[i]) and _indent(lines[i]) > parent_indent:
        i += 1  # skip option lines such as ":maxdepth: 2" or ":caption: ..."
    body = []
    while i < len(lines) and (not lines[i].strip() or _indent(lines[i]) > parent_indent):
        body.append(lines[i])
        i += 1
    while body and not body[-1].strip():  # trailing blank lines belong to what follows
        body.pop()
        i -= 1
    nonblank = [l for l in body if l.strip()]
    if nonblank:
        cut = min(_indent(l) for l in nonblank)
        body = [l[cut:] if l.strip() else "" for l in body]
    while body and not body[0].strip():
        body.pop(0)
    return body, i


def clean_rst(text: str) -> str:
    lines = text.splitlines()
    out: list[str] = []
    level_of: dict[str, int] = {}  # underline char -> heading level, in order of first appearance
    i = 0
    while i < len(lines):
        line = lines[i]

        # headings: a text line followed by an underline at least as long
        if (i + 1 < len(lines) and line.strip() and not line.startswith(" ")
                and UNDERLINE.match(lines[i + 1]) and len(lines[i + 1].strip()) >= len(line.strip())):
            ch = lines[i + 1].strip()[0]
            level = level_of.setdefault(ch, len(level_of) + 1)
            out += ["", "#" * min(level, 4) + " " + line.strip(), ""]
            i += 2
            continue

        # transitions ("----" with no heading text above it)
        if UNDERLINE.match(line):
            i += 1
            continue

        # link targets (".. _x:"), substitutions, comments: drop with their body
        if line.startswith(".. ") and not DIRECTIVE.match(line):
            _, i = _block(lines, i + 1, 0)
            continue

        m = DIRECTIVE.match(line)
        if m:
            name, arg = m.group(2), m.group(3)
            body, i = _block(lines, i + 1, len(m.group(1)))
            if name not in CODE_DIRECTIVES and body:  # bodies may hold nested directives / roles
                body = clean_rst("\n".join(body)).splitlines()
            if name in CODE_DIRECTIVES:
                out += ["", "```" + arg, *body, "```", ""]
            elif name in ADMONITIONS:
                out += ["", f"{ADMONITIONS[name]}: {arg}".rstrip(), *body, ""]
            elif name == "admonition":
                out += ["", f"{arg}:", *body, ""]
            elif name in VERSION_DIRECTIVES:
                out += ["", f"{VERSION_DIRECTIVES[name]} {arg}." + (" " + " ".join(body) if body else ""), ""]
            elif name in REFERENCE_DIRECTIVES:
                out += ["", arg.strip(), *body, ""]
            elif name in KEEP_BODY_DIRECTIVES:
                out += ["", *([arg] if arg else []), *body, ""]
            elif name in AUTODOC_DIRECTIVES or name in DROP_DIRECTIVES:
                pass
            else:  # unknown directive: keep its text, drop the marker
                out += ["", *body, ""]
            continue

        # literal blocks: "text::" followed by an indented block
        if line.rstrip().endswith("::"):
            head = line.rstrip()[:-2].rstrip()
            body, j = _block(lines, i + 1, _indent(line))
            out += ([head + ":"] if head else [])
            if body:
                out += ["", "```", *body, "```", ""]
                i = j
            else:
                i += 1
            continue

        out.append(line)
        i += 1

    # inline markup, but never inside code fences
    parts = re.split(r"(```.*?```)", "\n".join(out), flags=re.S)
    for k in range(0, len(parts), 2):
        parts[k] = _inline(parts[k])
    return re.sub(r"\n{3,}", "\n\n", "".join(parts)).strip() + "\n"

## 4. Before / after

In [6]:
def show(page, start_marker, n=900):
    """Print a slice of a raw page and the same slice after clean_rst()."""
    raw = (RAW_DIR / f"{page}.rst").read_text(encoding="utf-8")
    i = raw.index(start_marker)
    piece = raw[i:i + n]
    print("=" * 20, "RAW", page); print(piece)
    print("=" * 20, "CLEAN"); print(clean_rst(piece))

show("blueprints", "A :class:`Blueprint` object", 700)

==================== RAW blueprints
A :class:`Blueprint` object works similarly to a :class:`Flask`
application object, but it is not actually an application.  Rather it is a
*blueprint* of how to construct or extend an application.

Why Blueprints?
---------------

Blueprints in Flask are intended for these cases:

* Factor an application into a set of blueprints.  This is ideal for
  larger applications; a project could instantiate an application object,
  initialize several extensions, and register a collection of blueprints.
* Register a blueprint on an application at a URL prefix and/or subdomain.
  Parameters in the URL prefix/subdomain become common view arguments
  (with defaults) across all view functions in the bluep
==================== CLEAN
A Blueprint object works similarly to a Flask
application object, but it is not actually an application.  Rather it is a
*blueprint* of how to construct or extend an application.

# Why Blueprints?

Blueprints in Flask are intended for 

In [7]:
show("config", ".. py:data:: DEBUG", 900)

==================== RAW config
.. py:data:: DEBUG

    Whether debug mode is enabled. When using ``flask run`` to start the development
    server, an interactive debugger will be shown for unhandled exceptions, and the
    server will be reloaded when code changes. The :attr:`~flask.Flask.debug` attribute
    maps to this config key. This is set with the ``FLASK_DEBUG`` environment variable.
    It may not behave as expected if set in code.

    **Do not enable debug mode when deploying in production.**

    Default: ``False``

.. py:data:: TESTING

    Enable testing mode. Exceptions are propagated rather than handled by the
    the app's error handlers. Extensions may also change their behavior to
    facilitate easier testing. You should enable this in your own tests.

    Default: ``False``

.. py:data:: PROPAGATE_EXCEPTIONS

    Exceptions are re-raised rather than being handled by the app's error
    handlers. 
==================== CLEAN
DEBUG
Whether debug mode is enabled. Whe

In [8]:
show("quickstart", ".. code-block:: python", 350)

==================== RAW quickstart
.. code-block:: python

    from flask import Flask

    app = Flask(__name__)

    @app.route("/")
    def hello_world():
        return "<p>Hello, World!</p>"

So what did that code do?

1.  First we imported the :class:`~flask.Flask` class. An instance of
    this class will be our WSGI application.
2.  Next we create an instance of this class. 
==================== CLEAN
```python
from flask import Flask

app = Flask(__name__)

@app.route("/")
def hello_world():
    return "<p>Hello, World!</p>"
```

So what did that code do?

1.  First we imported the Flask class. An instance of
    this class will be our WSGI application.
2.  Next we create an instance of this class.



## 5. Sanity checks

After cleaning, scan every page for markup that survived (outside code fences, where raw syntax is legitimate).

In [9]:
CHECKS = {
    "directive (.. x::)": r"^\s*\.\. [\w:-]+::",
    "role (:x:`...`)":    r":[\w:-]+:`",
    "double backticks":   r"(?<!`)``(?!`)",
    "underline rows":     r"^[=\-~^]{4,}$",
    "trailing '::'":      r"::\s*$",
}
cleaned = {}
rows = []
for m in manifest:
    text = clean_rst((ROOT / m["path"]).read_text(encoding="utf-8"))
    cleaned[m["page"]] = text
    outside = re.sub(r"```.*?```", "", text, flags=re.S)  # ignore code fences
    rows.append({"page": m["page"], "raw_words": m["words"], "clean_words": len(text.split()),
                 **{k: len(re.findall(p, outside, re.M)) for k, p in CHECKS.items()}})
check = pd.DataFrame(rows)
print("leftover markup per check (want all 0):")
print(check[list(CHECKS)].sum().to_string())
print(f"\nwords: {check.raw_words.sum():,} -> {check.clean_words.sum():,}")

leftover markup per check (want all 0):
directive (.. x::)    0
role (:x:`...`)       0
double backticks      0
underline rows        0
trailing '::'         0

words: 59,570 -> 58,425


In [10]:
# pages that lost the most text (the API page loses its autodoc name lists)
check.assign(lost=check.raw_words - check.clean_words).sort_values("lost", ascending=False).head(6)[["page", "raw_words", "clean_words", "lost"]]

,page,raw_words,clean_words,lost
0,api,2593,2339,254
4,cli,2212,2139,73
67,tutorial/tests,2149,2087,62
22,index,257,199,58
61,tutorial/index,306,250,56
35,patterns/javascript,1198,1151,47


In [11]:
# every code fence must be opened and closed
unbalanced = [p for p, t in cleaned.items() if t.count("```") % 2]
print("pages with unbalanced code fences:", unbalanced)

pages with unbalanced code fences: []


## 6. Save cleaned pages

`data/processed/<page>.md` plus `manifest.jsonl` (same fields as the raw manifest, with the cleaned word count).
Next step: chunk these files.

In [12]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_manifest = []
for m in manifest:
    dest = PROCESSED_DIR / f"{m['page']}.md"
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(cleaned[m["page"]], encoding="utf-8")
    out_manifest.append({**m, "path": dest.relative_to(ROOT).as_posix(), "words": len(cleaned[m["page"]].split())})

with open(PROCESSED_DIR / "manifest.jsonl", "w", encoding="utf-8") as f:
    for m in out_manifest:
        f.write(json.dumps(m) + "\n")
print(f"saved {len(out_manifest)} pages, {sum(m['words'] for m in out_manifest):,} words -> {PROCESSED_DIR}")

saved 71 pages, 58,425 words -> D:\Coding Stuff\rag-under-hood\data\processed
